# Part 1.9 Stores

After this notebook you can keep facts across threads with a Store, and read them from
a node through `runtime.store`.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

A checkpointer remembers one thread. A Store remembers across threads, so a passenger's
third claim knows about the first two.

<img src="../../assets/slides/langgraph-memory-across-threads.jpg" width="720">

## Build it

In the finished graph, `check_history` reads a passenger's earlier claims and
`record_claim` writes each new one. File two claims, then read them back from the store.

In [ ]:
from langgraph.store.memory import InMemoryStore

from kea.notebook import claim, claims_graph

store = InMemoryStore()
graph = claims_graph(store)

for number, photo in enumerate(["wheel-01.jpg", "handle-02.jpg"]):
    graph.invoke(
        claim(photo, "pax-2001", f"KA30{number}", 300),
        {"configurable": {"thread_id": f"claim-2001-{number}"}},
    )

for item in store.search(("claims", "pax-2001")):
    print(item.key, item.value)

Two claims on file, so the third goes to review no matter how small.

In [ ]:
third = graph.invoke(
    claim("tear-02.jpg", "pax-2001", "KA302", 300),
    {"configurable": {"thread_id": "claim-2001-2"}},
)
third["__interrupt__"][0].value["reasons"]

Now build your own. A Store is a key-value store organized by **namespace**. Put an
item, then search the namespace.

In [ ]:
from langgraph.store.memory import InMemoryStore

demo = InMemoryStore()
demo.put(("claims", "pax-1"), "KA101", {"damage_type": "tear", "payout_usd": 100})
demo.put(("claims", "pax-1"), "KA102", {"damage_type": "crushed", "payout_usd": 200})

print([item.key for item in demo.search(("claims", "pax-1"))])
print(demo.get(("claims", "pax-1"), "KA101").value)

Nodes reach the store through `runtime.store`. Two different threads share it.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime


class State(TypedDict, total=False):
    passenger_id: str
    flight_number: str
    prior_claims: int


def remember(state: State, runtime: Runtime) -> dict:
    assert runtime.store is not None, "compile the graph with a store"
    namespace = ("claims", state["passenger_id"])
    prior = len(runtime.store.search(namespace))
    runtime.store.put(namespace, state["flight_number"], {"seen": True})
    return {"prior_claims": prior}


builder = StateGraph(State)
builder.add_node("remember", remember)
builder.add_edge(START, "remember")
builder.add_edge("remember", END)
memory_graph = builder.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

for thread, flight in enumerate(["KA1", "KA2", "KA3"]):
    result = memory_graph.invoke(
        {"passenger_id": "pax-9", "flight_number": flight},
        {"configurable": {"thread_id": f"t-{thread}"}},
    )
    print(f"thread t-{thread}: prior claims = {result['prior_claims']}")

## Under the hood

- **Checkpointer = one thread. Store = across threads.** Use the checkpointer for a
  conversation or a run in progress, and the Store for facts that outlive it.
- The namespace is a tuple, here `("claims", passenger_id)`. No thread is involved, so
  any run can read it.
- `record_claim` keys each item by flight number. Replaying a thread (time travel)
  overwrites the item instead of adding a second one.
- `InMemoryStore` is for the workshop. `langgraph dev` and LangSmith Deployment supply a
  persistent store, and the graph code doesn't change.

Docs: [Stores](https://docs.langchain.com/oss/python/langgraph/stores),
[Persistence](https://docs.langchain.com/oss/python/langgraph/persistence).

## Recap

- A Store holds facts across threads, keyed by namespace.
- Nodes use it through `runtime.store`; compile the graph with `store=`.
- Checkpointer for the run, Store for what lasts.

**Next:** [Part 1.10 Datasets](10_datasets.ipynb)